# BigQuery Analysis - Query Cloud Database

Testing BigQuery integration and comparing with pandas

### Libraries

In [1]:
import pandas as pd
from google.cloud import bigquery

### Set Up BigQuery Client

In [2]:
PROJECT_ID= 'motogp-racing-analytics'
client= bigquery.Client(project= PROJECT_ID)

print(f'Project: {client.project}')

Project: motogp-racing-analytics


### Count Races

In [3]:
query= ''' 
    SELECT COUNT(*) as total_races
    FROM motogp-racing-analytics.motogp_data.race_results_2024
 '''
result= client.query(query).to_dataframe()
print(result)

   total_races
0           23


### Wins by Rider

In [4]:
query= """
    SELECT 
        string_field_4 as winning_rider,
        COUNT(*) as total_wins
    FROM motogp-racing-analytics.motogp_data.race_results_2024
    GROUP BY string_field_4
    ORDER BY total_wins DESC
"""
df_wins= client.query(query).to_dataframe()
print(df_wins)

       winning_rider  total_wins
0  Francesco Bagnaia          11
1               None           4
2       Jorge Martín           3
3       Marc Márquez           3
4    Enea Bastianini           1
5   Maverick Viñales           1


### All Data

In [5]:
query= """
    SELECT * 
    FROM motogp-racing-analytics.motogp_data.race_results_2024
"""
df_bigquery= client.query(query).to_dataframe()

print(f"Rows: {len(df_bigquery)}")
print(f"Columns: {len(df_bigquery.columns)}")
print()
df_bigquery.head()

Rows: 23
Columns: 7



,string_field_0,string_field_1,string_field_2,string_field_3,string_field_4,string_field_5,string_field_6
0,–,7 April,Argentine Republic motorcycle Grand Prix,"Autódromo Termas de Río Hondo,Termas de Río Hondo",None,None,None
1,–,16 June22 September,Kazakhstan motorcycle Grand Prix,"Sokol International Racetrack,Almaty",None,None,None
2,–,22 September,Indian motorcycle Grand Prix,"Buddh International Circuit,Greater Noida",None,None,None
3,–,17 November,Valencian Community motorcycle Grand Prix,"Circuit Ricardo Tormo,Valencia",None,None,None
4,10,4 August,Monster Energy British Grand Prix,"Silverstone Circuit,Silverstone",Enea Bastianini,Aleix Espargaró,Aleix Espargaró


### Cloud Storage Method

In [6]:
from google.cloud import storage
from io import StringIO

storage_client= storage.Client(project= PROJECT_ID)
bucket= storage_client.bucket('motogp-racing-data-2024')
blob= bucket.blob('motogp_2024_results.csv')
csv_data= blob.download_as_text()
df_storage= pd.read_csv(StringIO(csv_data))

print(f"BigQuery: {len(df_bigquery)} rows, {len(df_bigquery.columns)} columns")
print(f"Cloud Storage: {len(df_storage)} rows, {len(df_storage.columns)} columns")

BigQuery: 23 rows, 7 columns
Cloud Storage: 23 rows, 7 columns
